Build schema_docs folder and docs

In [1]:
import os
os.makedirs('./schema_docs', exist_ok=True)

Write schema file

In [2]:
schema_doc = """
# Financial Transactions Database — Schema Reference

## Database Overview
A banking dataset covering transactions from 2010s, containing 2,000 customers,
6,146 cards, and 13M+ transactions. Fraud labels available for ~8.9M transactions.
Fraud rate is approximately 0.15% (highly imbalanced).

---

## Table: transactions
**Grain**: One row = one transaction  
**Row count**: ~13.3M  
**Primary key**: id

| Column         | Type    | Description |
|----------------|---------|-------------|
| id             | int     | Unique transaction ID |
| date           | string  | Timestamp of transaction (YYYY-MM-DD HH:MM:SS) |
| client_id      | int     | FK → users.id |
| card_id        | int     | FK → cards.id |
| amount         | float   | Transaction amount in USD (negative = refund) |
| use_chip       | string  | Payment method: 'Swipe Transaction', 'Chip Transaction', 'Online Transaction' |
| merchant_id    | int     | Merchant identifier |
| merchant_city  | string  | City where transaction occurred |
| merchant_state | string  | State where transaction occurred (2-letter code) |
| zip            | float   | Merchant ZIP code |
| mcc            | int     | FK → mcc_codes.mcc (merchant category code) |
| errors         | string  | Transaction error flags, NULL if none |

**Gotchas**:
- `amount` can be negative (refunds/reversals) — use `amount > 0` for spend analysis
- `errors` is NULL for most rows — don't filter it out unless specifically needed
- `date` is a string, cast with `CAST(date AS TIMESTAMP)` for date math

---

## Table: users
**Grain**: One row = one customer  
**Row count**: 2,000  
**Primary key**: id

| Column           | Type   | Description |
|------------------|--------|-------------|
| id               | int    | Unique user ID |
| current_age      | int    | Customer's current age |
| retirement_age   | int    | Expected retirement age |
| birth_year       | int    | Year of birth |
| birth_month      | int    | Month of birth |
| gender           | string | 'Male' or 'Female' |
| address          | string | Street address |
| latitude         | float  | Home location latitude |
| longitude        | float  | Home location longitude |
| per_capita_income| float  | Per capita income in USD (already cleaned, no $ sign) |
| yearly_income    | float  | Annual income in USD |
| total_debt       | float  | Total debt in USD |
| credit_score     | int    | Credit score (300–850 range) |
| num_credit_cards | int    | Number of credit cards held |

---

## Table: cards
**Grain**: One row = one card (users can have multiple cards)  
**Row count**: 6,146  
**Primary key**: id

| Column               | Type   | Description |
|----------------------|--------|-------------|
| id                   | int    | Unique card ID |
| client_id            | int    | FK → users.id |
| card_brand           | string | 'Visa', 'Mastercard', 'Amex', 'Discover' |
| card_type            | string | 'Debit', 'Credit' |
| card_number          | int    | Card number (synthetic) |
| expires              | string | Expiry date MM/YYYY |
| cvv                  | int    | CVV code |
| has_chip             | string | 'YES' or 'NO' |
| num_cards_issued     | int    | Number of cards issued on this account |
| credit_limit         | float  | Credit limit in USD (already cleaned, no $ sign) |
| acct_open_date       | string | Account open date MM/YYYY |
| year_pin_last_changed| int    | Year PIN was last changed |
| card_on_dark_web     | string | 'Yes' or 'No' — whether card appears on dark web |

**Gotchas**:
- One user can have multiple cards — JOIN on cards.client_id = users.id
- `card_on_dark_web = 'Yes'` is a high-risk signal worth flagging in fraud analysis

---

## Table: mcc_codes
**Grain**: One row = one merchant category  
**Row count**: 109  
**Primary key**: mcc

| Column      | Type   | Description |
|-------------|--------|-------------|
| mcc         | int    | Merchant Category Code |
| description | string | Human-readable category name (e.g. 'Eating Places and Restaurants') |

---

## Table: fraud_labels
**Grain**: One row = one labeled transaction  
**Row count**: ~8.9M (training set only, not all transactions are labeled)  
**Primary key**: transaction_id

| Column         | Type | Description |
|----------------|------|-------------|
| transaction_id | int  | FK → transactions.id |
| is_fraud       | int  | 1 = fraudulent, 0 = legitimate |

**Gotchas**:
- Only ~67% of transactions have labels — always use INNER JOIN (not LEFT JOIN)
  when fraud label is required, to avoid NULL confusion
- Fraud rate is 0.15% — when calculating fraud rates use COUNT + AVG(is_fraud),
  not SUM, to avoid confusion

---

## Common Join Patterns

### Full enriched transaction view
```sql
SELECT
    t.*,
    u.gender, u.credit_score, u.yearly_income,
    c.card_brand, c.card_type, c.card_on_dark_web,
    m.description AS merchant_category,
    fl.is_fraud
FROM transactions t
JOIN users u         ON t.client_id = u.id
JOIN cards c         ON t.card_id   = c.id
JOIN mcc_codes m     ON t.mcc       = m.mcc
JOIN fraud_labels fl ON t.id        = fl.transaction_id
```

### Fraud rate by merchant category
```sql
SELECT
    m.description,
    COUNT(*)              AS total_txns,
    SUM(fl.is_fraud)      AS fraud_count,
    AVG(fl.is_fraud)*100  AS fraud_rate_pct
FROM transactions t
JOIN mcc_codes m     ON t.mcc = m.mcc
JOIN fraud_labels fl ON t.id  = fl.transaction_id
GROUP BY m.description
ORDER BY fraud_rate_pct DESC
```
"""

with open('./schema_docs/schema.md', 'w') as f:
    f.write(schema_doc)

print("✓ schema_docs/schema.md is DONE")

✓ schema_docs/schema.md is DONE


Write Basic Agent

In [ ]:
import anthropic
import duckdb
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
con = duckdb.connect('/Users/isabella/data/fraud_analytics.db')

In [7]:
# Read schema file
with open('./schema_docs/schema.md', 'r') as f:
    schema_context = f.read()

def run_query(sql: str) -> str:
    """Execute SQL，return result strings"""
    try:
        result = con.execute(sql).df()
        if result.empty:
            return "Query returned no results."
        return result.to_string(index=False)
    except Exception as e:
        return f"SQL ERROR: {str(e)}"

def ask_agent(user_question: str) -> str:
    """Main Agent：Turn Natural Language to SQL and Execute"""
    
    system_prompt = f"""You are a data analyst for a banking fraud analytics platform.
You have access to a DuckDB database. Given a user question, write and execute SQL to answer it.

Always follow this process:
1. Think about which tables are needed
2. Write the SQL query
3. Present results clearly with brief business insight

{schema_context}

Rules:
- Always wrap SQL in ```sql ... ``` blocks
- For fraud analysis, remember fraud rate is only 0.15% — small absolute numbers are normal
- amount can be negative (refunds) — filter amount > 0 for spend analysis
- If asked about merchant types, JOIN with mcc_codes to get human-readable names
"""

    # Step 1: Claude Create SQL
    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        system=system_prompt,
        messages=[{"role": "user", "content": user_question}]
    )
    
    agent_reply = response.content[0].text
    
    # Step 2: Extract SQL and Execute
    import re
    sql_match = re.search(r'```sql\s*(.*?)\s*```', agent_reply, re.DOTALL)
    
    if not sql_match:
        return f"Agent reply (no SQL found):\n{agent_reply}"
    
    sql = sql_match.group(1).strip()
    query_result = run_query(sql)
    
    # Step 3: Claude Read Result
    interpretation = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=512,
        system=system_prompt,
        messages=[
            {"role": "user", "content": user_question},
            {"role": "assistant", "content": agent_reply},
            {"role": "user", "content": f"Query result:\n{query_result}\n\nPlease summarize the findings in 2-3 sentences."}
        ]
    )
    
    final_answer = interpretation.content[0].text
    
    # Print Whole Process（Easy for debug）
    print("─" * 60)
    print(f"QUESTION: {user_question}")
    print(f"\nSQL:\n{sql}")
    print(f"\nRAW RESULT:\n{query_result[:500]}")  # Only shows first 500 Chars
    print(f"\nINSIGHT:\n{final_answer}")
    print("─" * 60)
    
    return final_answer

Test

In [8]:
#Test a few questions, from easy to hard
ask_agent("What are the top 5 merchant categories by total transaction volume?")

────────────────────────────────────────────────────────────
QUESTION: What are the top 5 merchant categories by total transaction volume?

SQL:
SELECT
    m.description AS merchant_category,
    COUNT(*) AS total_transactions,
    SUM(t.amount) AS total_volume_usd,
    AVG(t.amount) AS avg_transaction_usd
FROM transactions t
JOIN mcc_codes m ON t.mcc = m.mcc
WHERE t.amount > 0  -- Exclude refunds/reversals
GROUP BY m.description
ORDER BY total_volume_usd DESC
LIMIT 5

RAW RESULT:
           merchant_category  total_transactions  total_volume_usd  avg_transaction_usd
              Money Transfer              578530       53158515.64            91.885495
            Service Stations             1135638       51239755.42            45.119796
Grocery Stores, Supermarkets             1592560       40970767.06            25.726357
             Wholesale Clubs              601928       37697568.71            62.628036
   Miscellaneous Food Stores             1170328       37352

INSIGHT:
**S

'**Summary:**\n\nMoney transfer services lead with **$53.2M** in total volume despite having fewer transactions (579K), indicating higher average ticket sizes at ~$92 per transaction. Service stations and grocery stores follow with $51.2M and $41M respectively, driven by high transaction frequency (1.1M+ and 1.6M transactions). These top 5 categories represent essential spending and financial services, collectively accounting for over $220M in transaction volume across the dataset.'

In [9]:
ask_agent("Which US states have the highest fraud rates?")

────────────────────────────────────────────────────────────
QUESTION: Which US states have the highest fraud rates?

SQL:
SELECT
    t.merchant_state,
    COUNT(*) AS total_transactions,
    SUM(fl.is_fraud) AS fraud_count,
    AVG(fl.is_fraud) * 100 AS fraud_rate_pct
FROM transactions t
JOIN fraud_labels fl ON t.id = fl.transaction_id
WHERE t.merchant_state IS NOT NULL
GROUP BY t.merchant_state
ORDER BY fraud_rate_pct DESC
LIMIT 15;

RAW RESULT:
merchant_state  total_transactions  fraud_count  fraud_rate_pct
        Tuvalu                   5          5.0      100.000000
         Haiti                 264        253.0       95.833333
         Italy                4706       3061.0       65.044624
            OH              324098        316.0        0.097501
            SD               20875          8.0        0.038323
            IA              107765         32.0        0.029694
            MO              131274         32.0     

INSIGHT:
**Key Findings:**

The data reveals t

'**Key Findings:**\n\nThe data reveals that international locations (Tuvalu, Haiti, Italy) have dramatically higher fraud rates (65-100%), suggesting cross-border transactions are major fraud vectors. Among US states, Ohio leads domestic fraud at 0.10%, but all US states show rates well below 0.10% - far lower than initially calculated and indicating that domestic transactions are relatively secure. The stark contrast between international (65%+) and domestic (<0.10%) fraud rates suggests the platform should prioritize enhanced screening for cross-border merchants.'

In [10]:
ask_agent("Do customers with cards on the dark web have higher fraud rates than others?")

────────────────────────────────────────────────────────────
QUESTION: Do customers with cards on the dark web have higher fraud rates than others?

SQL:
SELECT
    c.card_on_dark_web,
    COUNT(DISTINCT t.id) AS total_transactions,
    COUNT(DISTINCT CASE WHEN fl.is_fraud = 1 THEN t.id END) AS fraud_transactions,
    AVG(fl.is_fraud) * 100 AS fraud_rate_pct,
    COUNT(DISTINCT c.client_id) AS unique_customers,
    COUNT(DISTINCT c.id) AS unique_cards
FROM transactions t
JOIN cards c ON t.card_id = c.id
JOIN fraud_labels fl ON t.id = fl.transaction_id
GROUP BY c.card_on_dark_web
ORDER BY fraud_rate_pct DESC

RAW RESULT:
card_on_dark_web  total_transactions  fraud_transactions  fraud_rate_pct  unique_customers  unique_cards
              No             8914963               13332        0.149546              1219          4070

INSIGHT:
Based on the actual data, **only cards NOT on the dark web appear in the labeled transaction dataset** (8.9M transactions with 0.15% fraud rate).

This 

"Based on the actual data, **only cards NOT on the dark web appear in the labeled transaction dataset** (8.9M transactions with 0.15% fraud rate).\n\nThis suggests either: (1) dark web-exposed cards were excluded from the fraud labeling process, (2) they were proactively blocked/reissued before generating labeled transactions, or (3) the dark web flag may have been added after the fraud labeling period.\n\n**We cannot determine if dark web cards have higher fraud rates from this data** — we'd need labeled transactions for cards where `card_on_dark_web = 'Yes'` to make that comparison."

Add Adversarial Reviewer

In [11]:
def review_sql(sql: str, user_question: str) -> dict:
    """
    Adversarial reviewer：Specifically looking for SQL issues
    Return {"approved": True/False, "issues": [...], "fixed_sql": "..."}
    """
    
    reviewer_prompt = f"""You are an adversarial SQL reviewer for a banking fraud analytics database.
Your job is to find problems in SQL queries before they return wrong answers.

{schema_context}

Review this SQL query for the following question:
QUESTION: {user_question}
SQL:
```sql
{sql}
```

Check for these issues:
1. **Wrong join type** — should use INNER JOIN when fraud_labels is involved (not LEFT JOIN)
2. **Missing amount filter** — spend analysis should have `amount > 0` to exclude refunds
3. **Aggregation errors** — AVG(is_fraud) for fraud rate, not SUM
4. **Wrong table used** — e.g. using raw transactions when fraud_labels join is needed
5. **NULL handling** — merchant_state includes international codes, may need filtering
6. **Ambiguous column** — check all column names exist in the schema

Respond in this exact JSON format:
{{
  "approved": true or false,
  "issues": ["issue 1", "issue 2"],
  "fixed_sql": "corrected SQL here, or same SQL if no issues",
  "confidence": "high/medium/low"
}}

Be strict. If there are any issues, set approved to false and provide fixed_sql."""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        messages=[{"role": "user", "content": reviewer_prompt}]
    )
    
    import json, re
    raw = response.content[0].text
    
    # Extract JSON
    json_match = re.search(r'\{.*\}', raw, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except:
            pass
    
    # If parsing fails, default to approving.
    return {"approved": True, "issues": [], "fixed_sql": sql, "confidence": "low"}


def ask_agent_v2(user_question: str) -> str:
    """
    Agent v2：Add adversarial review Loop
    """
    import re
    
    system_prompt = f"""You are a data analyst for a banking fraud analytics platform.
You have access to a DuckDB database. Given a user question, write SQL to answer it.

{schema_context}

Rules:
- Always wrap SQL in ```sql ... ``` blocks
- amount can be negative (refunds) — filter amount > 0 for spend analysis
- For fraud rate: use AVG(is_fraud)*100, not SUM
- Always INNER JOIN fraud_labels (never LEFT JOIN)
"""

    # Step 1: Generate SQL
    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        system=system_prompt,
        messages=[{"role": "user", "content": user_question}]
    )
    agent_reply = response.content[0].text
    sql_match = re.search(r'```sql\s*(.*?)\s*```', agent_reply, re.DOTALL)
    
    if not sql_match:
        return "No SQL generated."
    
    sql = sql_match.group(1).strip()
    print(f"📝 Original SQL:\n{sql}\n")
    
    # Step 2: Adversarial review
    review = review_sql(sql, user_question)
    print(f"🔍 Review result:")
    print(f"   Approved: {review['approved']}")
    print(f"   Issues:   {review['issues']}")
    print(f"   Confidence: {review.get('confidence', 'N/A')}\n")
    
    # Step 3: If there's issue，use corrected SQL
    final_sql = sql if review['approved'] else review['fixed_sql']
    if not review['approved']:
        print(f"🔧 Fixed SQL:\n{final_sql}\n")
    
    # Step 4: Execute SQL
    query_result = run_query(final_sql)
    print(f"📊 Raw Result:\n{query_result[:500]}\n")
    
    # Step 5: Interpret the Result
    interpretation = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=512,
        system=system_prompt,
        messages=[
            {"role": "user", "content": user_question},
            {"role": "assistant", "content": agent_reply},
            {"role": "user", "content": f"Query result:\n{query_result}\n\nSummarize findings in 2-3 sentences."}
        ]
    )
    
    final_answer = interpretation.content[0].text
    print(f"💡 Insight:\n{final_answer}")
    print("─" * 60)
    return final_answer

Add tests specifically designed to trigger the reviewer

In [13]:
# This issue easily causes the agent to make mistakes: using a LEFT JOIN and forgetting to apply the `amount > 0` filter.
ask_agent_v2("What is the average transaction amount by card type for non-fraudulent transactions?")

📝 Original SQL:
SELECT
    c.card_type,
    COUNT(*) AS total_transactions,
    ROUND(AVG(t.amount), 2) AS avg_transaction_amount,
    ROUND(MIN(t.amount), 2) AS min_amount,
    ROUND(MAX(t.amount), 2) AS max_amount
FROM transactions t
INNER JOIN cards c ON t.card_id = c.id
INNER JOIN fraud_labels fl ON t.id = fl.transaction_id
WHERE fl.is_fraud = 0
    AND t.amount > 0
GROUP BY c.card_type
ORDER BY avg_transaction_amount DESC

🔍 Review result:
   Approved: True
   Issues:   []
   Confidence: high

📊 Raw Result:
      card_type  total_transactions  avg_transaction_amount  min_amount  max_amount
         Credit             2601190                   64.40        0.01     6613.44
          Debit             5261624                   46.41        0.01     5591.73
Debit (Prepaid)              589340                   25.31        0.03      500.00

💡 Insight:
Based on the analysis of non-fraudulent transactions, **Credit cards have the highest average transaction amount at $64.40**, followed

'Based on the analysis of non-fraudulent transactions, **Credit cards have the highest average transaction amount at $64.40**, followed by regular Debit cards at $46.41, and Prepaid Debit cards at $25.31. This pattern makes intuitive sense: credit cards are typically used for larger purchases, while prepaid debit cards (often used for budgeting or as gift cards) show both the lowest average spend and a strict $500 maximum transaction limit. The significant volume difference is also notable, with regular debit cards accounting for nearly 5.3M transactions compared to 2.6M credit card transactions.'

In [14]:
# A standard scenario to verify that the reviewer correctly approves it.
ask_agent_v2("What is the fraud rate by card brand (Visa, Mastercard, etc)?")

📝 Original SQL:
SELECT
    c.card_brand,
    COUNT(*) AS total_txns,
    SUM(fl.is_fraud) AS fraud_count,
    AVG(fl.is_fraud) * 100 AS fraud_rate_pct
FROM transactions t
JOIN cards c ON t.card_id = c.id
JOIN fraud_labels fl ON t.id = fl.transaction_id
GROUP BY c.card_brand
ORDER BY fraud_rate_pct DESC

🔍 Review result:
   Approved: False
   Issues:   ['Missing amount filter — the query counts all transactions including refunds (negative amounts). For fraud rate analysis, should filter to amount > 0 to focus on actual purchases, not refunds/reversals.']
   Confidence: high

🔧 Fixed SQL:
SELECT
    c.card_brand,
    COUNT(*) AS total_txns,
    SUM(fl.is_fraud) AS fraud_count,
    AVG(fl.is_fraud) * 100 AS fraud_rate_pct
FROM transactions t
JOIN cards c ON t.card_id = c.id
JOIN fraud_labels fl ON t.id = fl.transaction_id
WHERE t.amount > 0
GROUP BY c.card_brand
ORDER BY fraud_rate_pct DESC

📊 Raw Result:
card_brand  total_txns  fraud_count  fraud_rate_pct
  Discover      215014        46

'Discover cards have the highest fraud rate at 0.22%, followed by Amex at 0.16%, while Visa and Mastercard are slightly lower at around 0.15%. However, the differences are relatively small (less than 0.07 percentage points), suggesting that card brand itself is not a strong predictor of fraud. Visa and Mastercard account for the vast majority of transactions (~87% combined) and also have the most fraud cases in absolute numbers.'